## Representation learning baselines

Frozen **wav2vec 2.0**, **HuBERT**, and **Whisper** (encoder only) embeddings with a linear probe (**StandardScaler + LogisticRegression**) and **GroupKFold** grouped like the other model notebooks (`file_stem` for Androids, `participant_id` for RADAR). Metrics are saved under `results/metrics/repr_learn/`.

**Dependencies**: install once in a terminal (`pip install -r requirements.txt` or `pip install "transformers>=4.40,<5"`) so you get **transformers 4.x** (v5 is untested here). **Do not run `pip install` inside this notebook:** upgrading packages in a live Jupyter kernel often **crashes the kernel on Windows** because NumPy/Torch DLLs get out of sync. After any env change, use **Restart Kernel** before running model code.

If the kernel still dies: set `FORCE_CPU = True` in the code cell (GPU VRAM), set `BATCH_SIZE = 1`, and close other GPU apps.

**RADAR audio**: the processed RADAR CSV only stores `File` (e.g. `20201230_1100-scripted-1-1.wav`). Set `RADAR_AUDIO_ROOT` below to a folder whose tree contains those `.wav` files (files are resolved by name). If wavs are not available, the RADAR block will skip missing files automatically.

**Cell 2 (below)** trains a **small PyTorch MLP head** on pooled pretrained **Wav2Vec2 or HuBERT** frame outputs (encoder **weights frozen**; only the head updates). Requires **cell 1** to have been run first (shared `DEVICE`, `load_waveform_mono`, `MODEL_IDS`, etc.). Results: `repr_learn/androids_*_dl_head_cv_folds.csv`.

**Cell 3** runs the **same linear probe** on Nick's precomputed RADAR embeddings (`data/processed/Nick/Nick_repr_learn/`), grouped by `participant_id`. Requires cell 1 first (`run_group_cv`). No wav files needed.

In [1]:
from __future__ import annotations

import gc
import os
from pathlib import Path
from typing import Callable

# Avoid oversubscribing CPU threads (occasionally unstable in Jupyter on Windows)
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")

import librosa
import numpy as np
import pandas as pd
import torch
from scipy.stats import wilcoxon
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from transformers import (
    HubertModel,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Model,
    WhisperModel,
    WhisperProcessor,
)

# ---------------------------------------------------------------------------
# Paths (match other notebooks)
# ---------------------------------------------------------------------------
PROJECT = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project")
ANDROID_CSV = PROJECT / "data/processed/androids_model_dataset_basic.csv"
RADAR_CSV = PROJECT / "data/processed/radar_model_dataset_raw_features.csv"
RESULTS_PATH = PROJECT / "results/metrics/repr_learn"
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# Point this at the directory tree that holds RADAR WAVs (recursive lookup by File name).
# Example guesses — adjust until `Path.exists()` resolves your data:
RADAR_AUDIO_ROOT = PROJECT / "datasets/RADAR-MDD"

# --- stability (kernel dies on GPU OOM / Windows DLL mismatch after pip in-notebook) ---
FORCE_CPU = False  # True if CUDA crashes or you need deterministic CPU-only runs
BATCH_SIZE = 1   # increase to 2–4 only if GPU memory allows

torch.set_num_threads(min(8, max(1, os.cpu_count() or 1)))
if not FORCE_CPU and torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass
else:
    DEVICE = torch.device("cpu")

TARGET_SR = 16000
MAX_AUDIO_SECONDS = 30
N_FOLDS = 5

MODEL_IDS = {
    "wav2vec2": "facebook/wav2vec2-base",
    "hubert": "facebook/hubert-base-ls960",
    "whisper": "openai/whisper-base",
}


def _mean_pool(hidden: torch.Tensor, mask: torch.Tensor | None) -> torch.Tensor:
    if mask is None:
        return hidden.mean(dim=1)
    m = mask.unsqueeze(-1).to(hidden.dtype)
    summed = (hidden * m).sum(dim=1)
    denom = m.sum(dim=1).clamp(min=1e-6)
    return summed / denom


def build_wav_filename_index(audio_root: Path) -> dict[str, Path]:
    if not audio_root.exists():
        return {}
    idx: dict[str, Path] = {}
    for p in audio_root.rglob("*.wav"):
        idx.setdefault(p.name, p)
        idx.setdefault(p.name.lower(), p)
    return idx


def load_waveform_mono(path: Path, target_sr: int) -> np.ndarray:
    wav, sr = librosa.load(str(path), sr=target_sr, mono=True)
    max_len = int(MAX_AUDIO_SECONDS * target_sr)
    if len(wav) > max_len:
        wav = wav[:max_len]
    return wav.astype(np.float32)


def encode_wav2vec_family(paths: list[Path], model_id: str, batch_size: int | None = None) -> np.ndarray:
    bs = BATCH_SIZE if batch_size is None else batch_size
    processor = Wav2Vec2FeatureExtractor.from_pretrained(model_id)
    if "hubert" in model_id.lower():
        model = HubertModel.from_pretrained(model_id, low_cpu_mem_usage=True)
    else:
        model = Wav2Vec2Model.from_pretrained(model_id, low_cpu_mem_usage=True)
    model.to(DEVICE)
    model.eval()

    out_list: list[np.ndarray] = []
    for start in range(0, len(paths), bs):
        batch_paths = paths[start : start + bs]
        waves = [load_waveform_mono(p, TARGET_SR) for p in batch_paths]
        feats = processor(
            waves,
            sampling_rate=TARGET_SR,
            padding=True,
            return_tensors="pt",
        )
        feats = {k: v.to(DEVICE) for k, v in feats.items()}
        with torch.inference_mode():
            outputs = model(**feats)
        pooled = _mean_pool(outputs.last_hidden_state, feats.get("attention_mask"))
        out_list.append(pooled.cpu().numpy())

    model.cpu()
    del model, processor
    gc.collect()
    torch.cuda.empty_cache()
    return np.vstack(out_list)


def encode_whisper(paths: list[Path], model_id: str, batch_size: int | None = None) -> np.ndarray:
    bs = BATCH_SIZE if batch_size is None else batch_size
    processor = WhisperProcessor.from_pretrained(model_id)
    model = WhisperModel.from_pretrained(model_id, low_cpu_mem_usage=True)
    model.to(DEVICE)
    model.eval()

    out_list: list[np.ndarray] = []
    for start in range(0, len(paths), bs):
        batch_paths = paths[start : start + bs]
        waves = [load_waveform_mono(p, TARGET_SR) for p in batch_paths]
        inputs = processor(
            waves,
            sampling_rate=TARGET_SR,
            return_tensors="pt",
            padding=True,
        )
        input_features = inputs.input_features.to(DEVICE)
        with torch.inference_mode():
            enc = model.encoder(input_features)
            hidden = enc.last_hidden_state
        pooled = hidden.mean(dim=1)
        out_list.append(pooled.cpu().numpy())

    model.cpu()
    del model, processor
    gc.collect()
    torch.cuda.empty_cache()
    return np.vstack(out_list)


ENCODERS: dict[str, Callable[[list[Path], str, int], np.ndarray]] = {
    "wav2vec2": encode_wav2vec_family,
    "hubert": encode_wav2vec_family,
    "whisper": encode_whisper,
}


def align_embeddings(paths: pd.Series, unique_paths: np.ndarray, mat: np.ndarray) -> np.ndarray:
    lookup = {str(p): i for i, p in enumerate(unique_paths)}
    idx = np.array([lookup[str(p)] for p in paths], dtype=np.int64)
    return mat[idx]


def run_group_cv(
    X: np.ndarray,
    y: np.ndarray,
    groups: np.ndarray,
    n_folds: int = N_FOLDS,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    gkf = GroupKFold(n_splits=n_folds)
    fold_rows: list[dict] = []

    for fold, (tr, te) in enumerate(gkf.split(X, y, groups=groups), start=1):
        pipe = Pipeline(
            [
                ("scale", StandardScaler()),
                (
                    "clf",
                    LogisticRegression(
                        max_iter=5000,
                        class_weight="balanced",
                        solver="lbfgs",
                        random_state=42,
                    ),
                ),
            ]
        )
        pipe.fit(X[tr], y[tr])

        pred = pipe.predict(X[te])
        proba = pipe.predict_proba(X[te])[:, 1]

        dummy = DummyClassifier(strategy="stratified", random_state=42)
        dummy.fit(X[tr], y[tr])
        p_dummy = dummy.predict_proba(X[te])[:, 1]

        e_m = np.abs(y[te] - proba)
        e_d = np.abs(y[te] - p_dummy)
        if len(e_m) >= 2 and not np.allclose(e_m, e_d):
            w_p = float(wilcoxon(e_m, e_d, zero_method="wilcox", mode="auto").pvalue)
        else:
            w_p = float("nan")

        fold_rows.append(
            {
                "fold": fold,
                "n_train": len(tr),
                "n_test": len(te),
                "accuracy": accuracy_score(y[te], pred),
                "f1": f1_score(y[te], pred, zero_division=0),
                "roc_auc": roc_auc_score(y[te], proba),
                "wilcoxon_p_vs_stratified_dummy": w_p,
            }
        )

    folds_df = pd.DataFrame(fold_rows)
    summary = pd.DataFrame(
        [
            {
                "subset": "all",
                "n_rows": len(y),
                "n_groups": len(np.unique(groups)),
                "accuracy_mean": folds_df["accuracy"].mean(),
                "accuracy_std": folds_df["accuracy"].std(),
                "f1_mean": folds_df["f1"].mean(),
                "f1_std": folds_df["f1"].std(),
                "roc_auc_mean": folds_df["roc_auc"].mean(),
                "roc_auc_std": folds_df["roc_auc"].std(),
            }
        ]
    )
    return folds_df, summary


def run_repr_for_dataset(
    dataset_key: str,
    manifest: pd.DataFrame,
    path_series: pd.Series,
    groups: pd.Series,
    y: pd.Series,
) -> None:
    valid = (
        path_series.notna()
        & groups.notna()
        & y.notna()
        & path_series.astype(str).str.len().gt(0)
    )
    df = manifest.loc[valid].copy()
    paths = path_series.loc[valid].astype(str)
    grp = groups.loc[valid].astype(str).values
    yt = y.loc[valid].astype(int).values

    uniq = pd.unique(paths)
    uniq_paths = np.array([Path(p) for p in uniq])

    print(f"{dataset_key}: {len(paths)} usable rows | {len(uniq_paths)} unique audio files")

    for model_name, encoder in ENCODERS.items():
        print(f"  -> embeddings: {model_name} ({MODEL_IDS[model_name]})")
        emb_uniq = encoder(uniq_paths.tolist(), MODEL_IDS[model_name])
        X = align_embeddings(paths, uniq, emb_uniq)

        folds_df, summary_df = run_group_cv(X, yt, grp)

        folds_path = RESULTS_PATH / f"{dataset_key}_{model_name}_cv_folds.csv"
        summary_path = RESULTS_PATH / f"{dataset_key}_{model_name}_summary.csv"
        folds_df.to_csv(folds_path, index=False)
        summary_df.to_csv(summary_path, index=False)
        print(f"     saved: {folds_path.name}, {summary_path.name}")


def prepare_androids() -> tuple[pd.DataFrame, pd.Series, pd.Series, pd.Series]:
    df = pd.read_csv(ANDROID_CSV)
    df["file_path"] = df["file_path"].astype(str).str.strip()
    df["file_stem"] = df["file_stem"].astype(str).str.strip()
    df["depressed"] = pd.to_numeric(df["depressed"], errors="coerce")
    path_series = df["file_path"].map(lambda p: Path(p))
    exists = path_series.map(lambda p: p.is_file())
    df = df.loc[exists].copy()
    path_series = path_series.loc[exists]
    return df, path_series, df["file_stem"], df["depressed"]


def prepare_radar(wav_index: dict[str, Path]) -> tuple[pd.DataFrame, pd.Series, pd.Series, pd.Series] | None:
    df = pd.read_csv(RADAR_CSV)
    df["participant_id"] = df["participant_id"].astype(str).str.strip()
    df["phq8_score"] = pd.to_numeric(df["phq8_score"], errors="coerce")
    df = df.dropna(subset=["phq8_score", "File", "participant_id"]).copy()
    df["depressed"] = (df["phq8_score"] >= 10).astype(int)
    names = df["File"].astype(str).str.strip()

    def resolve(name: str) -> Path | None:
        if not name:
            return None
        if name in wav_index:
            return wav_index[name]
        low = name.lower()
        return wav_index.get(low)

    audio_paths = names.map(resolve)
    n_ok = audio_paths.notna().sum()
    if n_ok == 0:
        print(
            "RADAR: could not resolve any WAV paths. Set RADAR_AUDIO_ROOT so it contains",
            "the recording files referenced in column 'File'.",
        )
        return None

    df = df.loc[audio_paths.notna()].copy()
    paths_series = audio_paths.loc[audio_paths.notna()].map(lambda p: Path(p))
    if n_ok < len(names):
        print(f"RADAR: using {n_ok} / {len(names)} rows with found audio files")

    return df, paths_series, df["participant_id"], df["depressed"]


# ---------------------------------------------------------------------------
# Run
# ---------------------------------------------------------------------------
android_df, a_paths, a_groups, a_y = prepare_androids()
run_repr_for_dataset("androids", android_df, a_paths, a_groups, a_y)

RADAR_IDX = build_wav_filename_index(RADAR_AUDIO_ROOT)
rad = prepare_radar(RADAR_IDX)
if rad is not None:
    radar_df, r_paths, r_groups, r_y = rad
    run_repr_for_dataset("radar", radar_df, r_paths, r_groups, r_y)

print("Done. Outputs in:", RESULTS_PATH)

androids: 224 usable rows | 115 unique audio files
  -> embeddings: wav2vec2 (facebook/wav2vec2-base)


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


     saved: androids_wav2vec2_cv_folds.csv, androids_wav2vec2_summary.csv
  -> embeddings: hubert (facebook/hubert-base-ls960)


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

     saved: androids_hubert_cv_folds.csv, androids_hubert_summary.csv
  -> embeddings: whisper (openai/whisper-base)


Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

     saved: androids_whisper_cv_folds.csv, androids_whisper_summary.csv
RADAR: could not resolve any WAV paths. Set RADAR_AUDIO_ROOT so it contains the recording files referenced in column 'File'.
Done. Outputs in: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\metrics\repr_learn


In [4]:
# ---------------------------------------------------------------------------
# Simple DL: frozen Wav2Vec2 / HuBERT trunk + trainable 2-layer MLP head
# No folds: single group-aware train/test split
#
# Run your first setup cell first so these already exist:
# DEVICE, MODEL_IDS, prepare_androids, load_waveform_mono,
# RESULTS_PATH, TARGET_SR
# ---------------------------------------------------------------------------

import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader, Dataset
from transformers import HubertModel, Wav2Vec2FeatureExtractor, Wav2Vec2Model


try:
    DEVICE, MODEL_IDS, prepare_androids, load_waveform_mono, RESULTS_PATH, TARGET_SR
except NameError as e:
    raise RuntimeError(
        "Run the previous notebook cell first: imports, paths, DEVICE, MODEL_IDS, "
        "prepare_androids, load_waveform_mono, RESULTS_PATH, TARGET_SR."
    ) from e


# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

DL_BACKBONE = "wav2vec2"  # change to "hubert" if needed
DL_EPOCHS = 8
DL_BATCH = 4
DL_LR = 1e-3
DL_NUM_WORKERS = 0  # keep 0 on Windows + Jupyter
TEST_SIZE = 0.2
RANDOM_STATE = 42


# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------

class FrozenEncoderClassifier(nn.Module):
    """
    Frozen Wav2Vec2 / HuBERT encoder + trainable MLP classification head.
    """

    def __init__(self, backbone_key: str) -> None:
        super().__init__()

        model_id = MODEL_IDS[backbone_key]

        if backbone_key == "hubert":
            self.encoder = HubertModel.from_pretrained(
                model_id,
                low_cpu_mem_usage=True,
            )
        else:
            self.encoder = Wav2Vec2Model.from_pretrained(
                model_id,
                low_cpu_mem_usage=True,
            )

        # Freeze representation model
        for p in self.encoder.parameters():
            p.requires_grad = False

        hidden_size = self.encoder.config.hidden_size

        self.head = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(hidden_size, 128),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(128, 1),
        )

    def forward(
        self,
        input_values: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        out = self.encoder(
            input_values=input_values,
            attention_mask=attention_mask,
        ).last_hidden_state

        # Do NOT use the raw audio attention mask for pooling here.
        # The encoder output is downsampled, so hidden length != raw audio length.
        pooled = out.mean(dim=1)

        logits = self.head(pooled).squeeze(-1)
        return logits


# ---------------------------------------------------------------------------
# Dataset + collate
# ---------------------------------------------------------------------------

class AudioDataset(Dataset):
    def __init__(self, path_strings: list[str], labels: np.ndarray) -> None:
        self.paths = path_strings
        self.labels = labels.astype(np.float32)

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int) -> tuple[np.ndarray, float]:
        wav = load_waveform_mono(Path(self.paths[idx]), TARGET_SR)
        label = float(self.labels[idx])
        return wav, label


def dl_collate(
    batch: list[tuple[np.ndarray, float]],
    processor,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    waves, labels = zip(*batch)
    wave_list = list(waves)

    try:
        feats = processor(
            wave_list,
            sampling_rate=TARGET_SR,
            padding=True,
            return_tensors="pt",
            return_attention_mask=True,
        )
    except TypeError:
        feats = processor(
            wave_list,
            sampling_rate=TARGET_SR,
            padding=True,
            return_tensors="pt",
        )

    input_values = feats["input_values"]

    if "attention_mask" in feats:
        attention_mask = feats["attention_mask"]
    else:
        lengths = [len(w) for w in wave_list]
        batch_size, max_len = input_values.shape

        attention_mask = torch.zeros(
            batch_size,
            max_len,
            dtype=torch.long,
        )

        for i, length in enumerate(lengths):
            attention_mask[i, : min(length, max_len)] = 1

    y = torch.tensor(labels, dtype=torch.float32)

    return input_values, attention_mask, y


# ---------------------------------------------------------------------------
# Train / evaluate helpers
# ---------------------------------------------------------------------------

def train_one_epoch(
    model: FrozenEncoderClassifier,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: nn.Module,
) -> float:
    model.train()

    total_loss = 0.0
    n_samples = 0

    for input_values, attention_mask, yb in loader:
        input_values = input_values.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer.zero_grad()

        logits = model(input_values, attention_mask)
        loss = loss_fn(logits, yb)

        loss.backward()
        optimizer.step()

        total_loss += float(loss.detach().cpu()) * yb.size(0)
        n_samples += yb.size(0)

    return total_loss / max(n_samples, 1)


@torch.inference_mode()
def evaluate_model(
    model: FrozenEncoderClassifier,
    loader: DataLoader,
) -> tuple[np.ndarray, np.ndarray]:
    model.eval()

    all_logits = []
    all_labels = []

    for input_values, attention_mask, yb in loader:
        input_values = input_values.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)

        logits = model(input_values, attention_mask)

        all_logits.append(logits.cpu().numpy())
        all_labels.append(yb.numpy())

    logits = np.concatenate(all_logits)
    labels = np.concatenate(all_labels)

    return logits, labels


# ---------------------------------------------------------------------------
# Main run function: no folds
# ---------------------------------------------------------------------------

def run_dl_androids_no_folds() -> None:
    df, paths, groups, y = prepare_androids()

    valid = paths.notna() & groups.notna() & y.notna()

    path_arr = paths.loc[valid].astype(str).values
    y_arr = y.loc[valid].astype(int).values
    group_arr = groups.loc[valid].astype(str).values

    print(f"Usable rows: {len(y_arr)}")
    print(f"Unique groups: {len(np.unique(group_arr))}")

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
    )

    train_idx, test_idx = next(
        splitter.split(
            np.zeros(len(y_arr)),
            y_arr,
            groups=group_arr,
        )
    )

    print(f"Train rows: {len(train_idx)}")
    print(f"Test rows: {len(test_idx)}")
    print(f"Train groups: {len(np.unique(group_arr[train_idx]))}")
    print(f"Test groups: {len(np.unique(group_arr[test_idx]))}")

    processor = Wav2Vec2FeatureExtractor.from_pretrained(
        MODEL_IDS[DL_BACKBONE]
    )

    train_ds = AudioDataset(
        path_arr[train_idx].tolist(),
        y_arr[train_idx],
    )

    test_ds = AudioDataset(
        path_arr[test_idx].tolist(),
        y_arr[test_idx],
    )

    collate_fn = lambda batch: dl_collate(batch, processor)

    train_loader = DataLoader(
        train_ds,
        batch_size=DL_BATCH,
        shuffle=True,
        num_workers=DL_NUM_WORKERS,
        collate_fn=collate_fn,
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=DL_BATCH,
        shuffle=False,
        num_workers=DL_NUM_WORKERS,
        collate_fn=collate_fn,
    )

    model = FrozenEncoderClassifier(DL_BACKBONE).to(DEVICE)

    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=DL_LR,
    )

    y_train = y_arr[train_idx]
    n_pos = int((y_train == 1).sum())
    n_neg = int((y_train == 0).sum())

    pos_weight = torch.tensor(
        [n_neg / max(n_pos, 1)],
        dtype=torch.float32,
        device=DEVICE,
    )

    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    for epoch in range(1, DL_EPOCHS + 1):
        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            loss_fn,
        )

        print(
            f"Epoch {epoch}/{DL_EPOCHS} "
            f"train_loss={train_loss:.4f}"
        )

    logits_test, y_test = evaluate_model(model, test_loader)

    proba = 1.0 / (1.0 + np.exp(-logits_test))
    pred = (proba >= 0.5).astype(int)

    accuracy = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred, zero_division=0)

    if len(np.unique(y_test)) > 1:
        roc_auc = roc_auc_score(y_test, proba)
    else:
        roc_auc = np.nan
        print("Warning: ROC-AUC is NaN because the test set has only one class.")

    results = {
        "backbone": DL_BACKBONE,
        "n_rows": len(y_arr),
        "n_train": len(train_idx),
        "n_test": len(test_idx),
        "n_groups": len(np.unique(group_arr)),
        "n_train_groups": len(np.unique(group_arr[train_idx])),
        "n_test_groups": len(np.unique(group_arr[test_idx])),
        "epochs": DL_EPOCHS,
        "batch_size": DL_BATCH,
        "learning_rate": DL_LR,
        "accuracy": accuracy,
        "f1": f1,
        "roc_auc": roc_auc,
    }

    results_df = pd.DataFrame([results])

    tag = f"androids_{DL_BACKBONE}_dl_head_no_folds"
    summary_path = RESULTS_PATH / f"{tag}_summary.csv"
    predictions_path = RESULTS_PATH / f"{tag}_predictions.csv"

    results_df.to_csv(summary_path, index=False)

    pred_df = pd.DataFrame(
        {
            "file_path": path_arr[test_idx],
            "group": group_arr[test_idx],
            "y_true": y_test.astype(int),
            "probability": proba,
            "prediction": pred.astype(int),
        }
    )

    pred_df.to_csv(predictions_path, index=False)

    print()
    print(results_df)
    print()
    print("Saved summary:", summary_path)
    print("Saved predictions:", predictions_path)

    model.cpu()
    del model, optimizer, loss_fn, train_loader, test_loader
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ---------------------------------------------------------------------------
# Run
# ---------------------------------------------------------------------------

run_dl_androids_no_folds()

Usable rows: 224
Unique groups: 115
Train rows: 179
Test rows: 45
Train groups: 92
Test groups: 23


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/8 train_loss=0.5883
Epoch 2/8 train_loss=0.4838
Epoch 3/8 train_loss=0.3829
Epoch 4/8 train_loss=0.3462
Epoch 5/8 train_loss=0.2641
Epoch 6/8 train_loss=0.2068
Epoch 7/8 train_loss=0.1908
Epoch 8/8 train_loss=0.1549

   backbone  n_rows  n_train  n_test  n_groups  n_train_groups  n_test_groups  \
0  wav2vec2     224      179      45       115              92             23   

   epochs  batch_size  learning_rate  accuracy        f1  roc_auc  
0       8           4          0.001  0.822222  0.826087  0.97619  

Saved summary: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\metrics\repr_learn\androids_wav2vec2_dl_head_no_folds_summary.csv
Saved predictions: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\metrics\repr_learn\androids_wav2vec2_dl_head_no_folds_predictions.csv


In [ ]:
# ---------------------------------------------------------------------------
# RADAR linear probe on Nick's precomputed embeddings (three encoders)
# Same evaluation as cell 1 (GroupKFold + StandardScaler + LogisticRegression).
# Does not re-encode audio.
#
# Run cell 1 first so these already exist:
# PROJECT, RESULTS_PATH, N_FOLDS, run_group_cv
# ---------------------------------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd

try:
    PROJECT, RESULTS_PATH, N_FOLDS, run_group_cv
except NameError as e:
    raise RuntimeError(
        "Run the first code cell first: PROJECT, RESULTS_PATH, N_FOLDS, run_group_cv."
    ) from e

# Nick's RADAR encoder embeddings (one CSV per site x encoder).
RADAR_EMBED_DIR = PROJECT / "data/processed/Nick/Nick_repr_learn"
if not RADAR_EMBED_DIR.is_dir():
    RADAR_EMBED_DIR = RESULTS_PATH

RADAR_SITES = ["KCL", "CIBER", "IISPV"]
RADAR_ENCODERS = ["wav2vec2", "hubert", "whisper"]


def prepare_radar_embeddings(embed_dir: Path, site: str, encoder: str) -> tuple[np.ndarray, np.ndarray, np.ndarray, pd.DataFrame]:
    path = embed_dir / f"radar_{site}_{encoder}_embeddings.csv"
    if not path.is_file():
        raise FileNotFoundError(f"Missing RADAR embeddings: {path}")

    df = pd.read_csv(path)
    df["participant_id"] = df["participant_id"].astype(str).str.strip()

    if "label" in df.columns:
        df["depressed"] = pd.to_numeric(df["label"], errors="coerce")
    else:
        df["depressed"] = (pd.to_numeric(df["phq8_score"], errors="coerce") >= 10).astype(float)

    emb_cols = [c for c in df.columns if c.startswith("emb_")]
    df = df.dropna(subset=["participant_id", "depressed"] + emb_cols).copy()
    df["depressed"] = df["depressed"].astype(int)

    X = df[emb_cols].to_numpy(dtype=np.float32)
    y = df["depressed"].to_numpy(dtype=int)
    groups = df["participant_id"].to_numpy(dtype=str)
    return X, y, groups, df


print("RADAR embeddings dir:", RADAR_EMBED_DIR)

for site in RADAR_SITES:
    for encoder in RADAR_ENCODERS:
        dataset_key = f"radar_{site}"
        print(f"  -> {dataset_key} embeddings: {encoder}")

        X, yt, grp, radar_df = prepare_radar_embeddings(RADAR_EMBED_DIR, site, encoder)
        print(f"     {dataset_key}: {len(yt)} usable rows | {len(np.unique(grp))} participants")

        folds_df, summary_df = run_group_cv(X, yt, grp)

        folds_path = RESULTS_PATH / f"{dataset_key}_{encoder}_cv_folds.csv"
        summary_path = RESULTS_PATH / f"{dataset_key}_{encoder}_summary.csv"
        folds_df.to_csv(folds_path, index=False)
        summary_df.to_csv(summary_path, index=False)
        print(f"     saved: {folds_path.name}, {summary_path.name}")

print("Done. RADAR probe outputs in:", RESULTS_PATH)